# 03 — Features & corrélations

Inspection du dataset features + targets construit par la Phase 2.

**Prérequis** : `python scripts/02_build_features.py` a tourné une fois pour produire `data/processed/features_targets.parquet`.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = (12, 5)
df = pd.read_parquet('../data/processed/features_targets.parquet')
feat_cols = [c for c in df.columns if c not in
             {'open','high','low','close','volume','y_reg_h24','y_clf_h24','y_clf_threshold'}]
print(f'Rows: {len(df):,}  |  Features: {len(feat_cols)}')
df.head()

## 1. Distribution des cibles

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(df['y_reg_h24'].dropna(), bins=200, color='steelblue', alpha=0.8)
axes[0].set_xlim(-0.05, 0.05); axes[0].set_title('y_reg (24h log-return)')
df['y_clf_h24'].value_counts().sort_index().plot(
    kind='bar', ax=axes[1], color=['firebrick','gray','steelblue'])
axes[1].set_title('y_clf — class balance'); plt.tight_layout(); plt.show()

## 2. Heatmap de corrélation

In [ ]:
fig, ax = plt.subplots(figsize=(14, 12))
sns.heatmap(df[feat_cols].corr(), cmap='RdBu_r', center=0, vmin=-1, vmax=1,
            ax=ax, cbar_kws={'shrink': 0.7})
plt.xticks(rotation=90, fontsize=8); plt.yticks(rotation=0, fontsize=8)
plt.title('Pearson correlation — features'); plt.tight_layout(); plt.show()

## 3. Corrélation avec la cible de régression

In [ ]:
corrs = df[feat_cols + ['y_reg_h24']].corr()['y_reg_h24'].drop('y_reg_h24').sort_values()
fig, ax = plt.subplots(figsize=(8, 10))
corrs.plot(kind='barh', ax=ax,
    color=corrs.apply(lambda x: 'firebrick' if x<0 else 'steelblue'))
ax.axvline(0, color='k', lw=0.5)
ax.set_title('Pearson — features vs y_reg'); ax.set_xlabel('ρ')
plt.tight_layout(); plt.show()

## 4. Information mutuelle

MI capte aussi les relations non-linéaires (contrairement à Pearson).
On sous-échantillonne 20k lignes pour la vitesse.

In [ ]:
from sklearn.feature_selection import mutual_info_regression, mutual_info_classif

clean = df[feat_cols + ['y_reg_h24','y_clf_h24']].dropna()
idx = np.random.RandomState(42).choice(len(clean), size=20000, replace=False)
sub = clean.iloc[idx]
mi_reg = mutual_info_regression(sub[feat_cols].values, sub['y_reg_h24'].values, random_state=0)
mi_clf = mutual_info_classif(sub[feat_cols].values, sub['y_clf_h24'].astype(int).values, random_state=0)
mi_df = pd.DataFrame({'MI_reg': mi_reg, 'MI_clf': mi_clf}, index=feat_cols).sort_values('MI_reg', ascending=False)
mi_df.head(15)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 10))
mi_df['MI_reg'].sort_values().plot(kind='barh', ax=axes[0], color='darkorange')
axes[0].set_title('Mutual information — y_reg')
mi_df['MI_clf'].sort_values().plot(kind='barh', ax=axes[1], color='seagreen')
axes[1].set_title('Mutual information — y_clf')
plt.tight_layout(); plt.show()

## 5. Vérification anti-leakage (manuelle)

In [ ]:
# La 1ère ligne de chaque feature doit être NaN (shift de 1 bougie)
first_row = df[feat_cols].iloc[0]
all_nan_first = first_row.isna().sum()
print(f'Features NaN at row 0: {all_nan_first}/{len(feat_cols)} (calendar features sont OK non-NaN)')

# Sanity check : y_reg(t) doit être calculable depuis close(t+24)
t = df.index[1000]
t_plus_h = df.index[1024]
expected = np.log(df.loc[t_plus_h, 'close'] / df.loc[t, 'close'])
actual = df.loc[t, 'y_reg_h24']
print(f'y_reg sanity check at {t}: actual={actual:.6f}, expected={expected:.6f}, equal={np.isclose(actual, expected)}')